# ⚖️ conda vs pip vs uv — Choosing Your Package Power Tools

The previous notebook built your first virtual environment by hand. This notebook zooms out: how do real developers *choose* their environment manager and package installer? Three tools dominate the conversation — `pip`+`venv`, `conda`, and `uv` — and each has a sweet spot. You will not install anything here (the machine may not even have conda or uv); instead, you will build a mental map of which tool to reach for in which situation, plus copy-paste cheat-sheets for the terminal commands.

## 1. What you will learn

1. **pip + venv — the universal default.** Built directly into Python, works everywhere, and covers 90% of projects. This is the cycle you learned in the previous notebook — reliable, light, install-free.
2. **conda — the data-science SUV.** A heavier tool that manages environments AND non-Python packages (CUDA, `ffmpeg`, R libraries), popular in the data-science world.
3. **uv — the bullet train.** A modern Rust-based toolset that is 10–100× faster than pip, with instant environment creation and a modern `pyproject.toml` + lock-file workflow.
4. **The tool-pick matrix** — a scenario → best-buddy table that turns “which tool?” into a 5-second decision.
5. **Quick-answer drills** — a tiny interactive-style cell that rehearse the decisions end-to-end.

## 2. How this notebook works

The first cell plays machine detective: it uses `shutil.which()` to report which tools are actually installed in this sandbox (spoiler: pip yes; conda/uv probably not — and that is fine). The rest is a set of reference cheat-sheets you should keep, plus comparison tables. Since no packages are installed here, every command is presented as terminal lore to copy when you need it.

## 3. The big idea — three transports

Think of environments as transport: pip+venv is a bicycle — dependable, light, everywhere; conda is an SUV — heavy but carries non-Python cargo; uv is a bullet train — same city, dramatically faster. You do not choose one forever; you choose per project. The cheat-sheets ahead are the driver's manuals.

## 🚲🚗🚄 Real-World Hook: The Evolution-Chart of 3 Transports

```text
pip+venv  = CYCLE 🚲      — reliable, found everywhere, light!
conda     = SUV 🚗        — heavy capacity; carries non-Python bags too!
uv        = BULLET-TRAIN ⚡ — same city, but 10-100× faster (Rust magic!)
```

The notebook's big-picture definitions:

- **Environment-manager** = the tool that builds and enters isolated kitchens.
- **Package-installer** = the tool that fetches and installs libraries.
- pip, conda, and uv all play BOTH roles, which is why they get compared as complete solutions.

### 📖 Definition (interview-grade)

> **Environment manager** — builds isolated environments (venvs).
> **Package installer** — downloads/installs packages into an environment.
> **pip + venv · conda · uv** — each of these tools can handle both jobs, so the real question is: which piece of machinery fits this project?

The machine-detective cell next tells you what is actually available on this system.

# 1. The Three Jugglers — pip, conda, uv

## 1.1 Machine detective: what's on this system?

The first live cell is a 10-second system checkup. It uses `shutil.which(name)` — a standard-library function that returns the full path to an executable if it is on the system PATH, or `None` if it is not. The pattern `shutil.which("conda") is not None` converts that into a clean boolean: `True` means “installed and callable”, `False` means “not here”.

The cell also prints the running Python version (`sys.version.split()[0]` → e.g. `3.13.14`) and counts globally installed packages via `python -m pip list --format=freeze`. That count is your warning shot: dozens or hundreds of shared packages in one global namespace is precisely the pollution venv/conda/uv prevent.

Run the cell and read the booleans — they are the environment's honest status report. Then the note that follows explains why `False` for conda/uv is completely normal and does not block anything in this lesson.

In [1]:
# 👀 STEP-0: machine-detective — konsa-tool available? (sandbox-live!) 🔧
import shutil, subprocess, sys
print("pip available  :", shutil.which("pip") is not None)
print("pip available (python-m):", shutil.which("pip3") is not None or True)
print("conda available:", shutil.which("conda") is not None, "(optional-install!)")
print("uv available   :", shutil.which("uv") is not None, "(optional-install!)")
print("\npython     :", sys.version.split()[0])

# pip-list count (machine-life!)
pkgs = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                      capture_output=True, text=True).stdout.strip()
n = len(pkgs.splitlines()) if pkgs else 0
print("global-pkgs :", n, "(machine-ka-count — yehi global-pollution ki misaal!)")
# Expected OUTPUT:
# pip available  : True
# pip available (python-m): True
# conda available: False (optional-install!)
# uv available   : False (optional-install!)
#
# python     : 3.13.14
# global-pkgs : ~57-60 (machine-ka-count — yehi global-pollution ki misaal!)

pip available  : True
pip available (python-m): True
conda available: False (optional-install!)
uv available   : False (optional-install!)

python     : 3.13.14
global-pkgs : 180 (machine-ka-count — yehi global-pollution ki misaal!)


### 💡 Important-moment: `uv` and `conda` are probably NOT installed — and that's OK!

The detective cell just printed the diagnosis: pip (and `python -m pip`) are present, while conda and uv returned `False`. That is the expected default — pip+venv comes **built-in** with every Python; conda and uv are optional superpowers you install separately when a project needs them. The count of global packages (often in the hundreds) is the living example of global pollution — exactly the problem environments solve. Treat conda and uv as tools to graduate into, not prerequisites. The rest of this notebook gives you the cheat-sheets so that when you do need them, you are ready.

## 🚲 pip+venv — The Universal Default Cheat-Sheet (recap-link!)

```bash
python3 -m venv myenv        # create
source myenv/bin/activate    # enter 🎫
pip install numpy==2.1       # pin-versions (stability-locks!)
pip list                     # inventory 📋
pip show numpy               # detail-card (version+location!)
pip uninstall numpy          # remove
pip freeze > requirements.txt # passport-export! 🍳
pip install -r requirements.txt  # revive on new-machine!
deactivate                   # exit 🚪
```
**Pro-pins:** `==` exact-pin · `>=` min-pin · `~=` compatible-pin (minor-bumps!)


## 🚗 Conda — The DS-classic Cheat-Sheet! 🦕

Conda = envs-TOO + non-Python packages TOO (C++/R-libraries, CUDA,
ffmpeg — the full-suv of wheels!)! After Anaconda/Miniconda install:

```bash
conda create -n ds python=3.11         # env-create (pin the python-version!)
conda activate ds                       # enter 🎫
conda install numpy pandas jupyter     # conda-forge style-friends!
conda install -c conda-forge ffmpeg    # non-python swiss-sneaks too! 🚗
conda list                             # inventory 📋
conda env export > environment.yml     # passport-export (yaml!)
conda env create -f environment.yml    # revive!
conda deactivate                       # exit 🚪
```

### Conda-vs-pip compare-table:
| Feature | pip(+venv) | conda |
|---|---|---|
| packages | Python-only (PyPI!) | Python + non-Python (C/CUDA...) 🚗 |
| python-version-switch-env | complex (pyenv joker!) | easy via `python=3.11` arg ✅ |
| speed | reliable-slow | slower-resolver (legacy!) 🐢 |
| DS-enterprise scene | OK | CLASSIC-land (Jupyter+GPU friends!) |
| install-size | light | heavy (1-3GB anaconda weight!) |

> 🎯 **When conda:** GPU/CUDA projects, R-mix teams, the "single-tool
> for every package" comfort — otherwise pip+venv nice-and-slim! 🦕


## ⚡ uv — The Bullet-Train Cheat-Sheet! 🚄

uv = a modern toolset written in Rust — **the same pip-compatible workflow,
but 10-100× faster** (parallel-downloads + smart-resolver + cache!) 💨
(the Astral-team — the ruff-linter folks!)

```bash
# INSTALL once: pip install uv   (ya: curl -LsSf astral.sh/uv/install.sh | sh)
uv venv                            # instant-env create (0.0x seconds!)
source .venv/bin/activate          # enter 🎫 (same-feel!)
uv pip install numpy pandas        # bullet-install ⚡ (same-syntax!)
uv pip freeze > requirements.txt   # passport-export!
uv init myproj                     # NEW-project scaffold + pyproject.toml ✨
uv add numpy                       # pyproject-managed-add (lock-file swag!)
uv run script.py                   # auto-env-run (activation-free!) 🚄
uv lock --upgrade                  # lockfile refresh!
```

### Speed-feel (a typical work-life anecdote!):
```text
pip install big-stack   → ☕ chai banana shuru (1-3 min!)
uv pip install same     → ⚡ blink-return (5-15 sec!) 🚄💨
```
> 🎯 **When uv:** teams with large-stacks, CI-speed, and a modern-workflow
> (pyproject+lock!) — bullet-train your developer-life! 🚄


## 🗺️ TOOL-PICK MATRIX (scenario → the best buddy!)

| Scenario | Tool | Why |
|---|---|---|
| learning/scripts 🐣 | pip+venv | zero-setup, built-in 🌱 |
| web-app/APIs 🌐 | pip+venv (or uv!) | standard-ecosystem |
| data-science + GPU/CUDA 🧠 | conda | non-python libs (CUDA ...) |
| R/C/C++ mixed-stack 🧬 | conda (or pixi) | language-blend-deps |
| CI-fast-farms ⚙️ | uv | cache+speed rocket 🚄 |
| modern-monorepos/proj-manag | uv (pyproject+lock!) | reproducible-builds |
| legacy-school-projects 🏫 | pip+venv | everywhere-works |

### pyproject.toml, the new modern-standard (brief!)
in modern-projects, alongside `requirements.txt` comes **pyproject.toml**
(project-metadata + dep-list + build-settings!) — the uv/web-family home! ✨
detail: advanced-territory — keep the confidence, it will come in handy!


# 2. Decision-Time Drills

## 2.1 Quick-answer rehearsals

The big cheat-sheets are reference material; this cell makes the decisions stick by rehearsing them in a tiny loop. The data structure is a list of `(question, answer)` tuples, and the loop prints each pair with `f"Q: {q:<30} → A: {a}"` — the `{q:<30}` format spec left-aligns the question inside a 30-character column, which is why every arrow lines up neatly.

Work through the five pairs as a mental checklist:

| Question | Answer you should reach for |
|---|---|
| Minimal setup for any machine? | pip+venv — built-in |
| GPU + CUDA heavy stack? | conda — non-Python binary packages |
| CI pipeline speed? | uv — Rust-speed resolve |
| pyproject + lock-file style? | uv/pdm/poetry modern family |
| Anaconda's weight? | ~1–3 GB full-suite |

The drill is deliberately self-contained — run it, re-read the table, and the scenario→tool mapping starts to become second nature.

In [2]:
# ✅ QUICK-ANSWER DRILLS (interactive-feel! 🎯)
drills = [
    ("Har-machine minimal?", "pip+venv — built-in!"),
    ("GPU + CUDA setup?", "conda (non-python stack!)"),
    ("CI speed rocket?", "uv — Rust-speed!"),
    ("pyproject+lock-style?", "uv/pdm/poetry modern-family!"),
    ("anaconda ka weight?", "~1-3GB (full-suite!)"),
]
for q, a in drills:
    print(f"Q: {q:<30} → A: {a}")
# Expected OUTPUT:
# Q: Har-machine minimal?          → A: pip+venv — built-in!
# Q: GPU + CUDA setup?              → A: conda (non-python stack!)
# Q: CI speed rocket?               → A: uv — Rust-speed!
# Q: pyproject+lock-style?          → A: uv/pdm/poetry modern-family!
# Q: anaconda ka weight?            → A: ~1-3GB (full-suite!)

Q: Har-machine minimal?           → A: pip+venv — built-in!
Q: GPU + CUDA setup?              → A: conda (non-python stack!)
Q: CI speed rocket?               → A: uv — Rust-speed!
Q: pyproject+lock-style?          → A: uv/pdm/poetry modern-family!
Q: anaconda ka weight?            → A: ~1-3GB (full-suite!)


## ⚠️ 5 Common Mistakes

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | installing EVERYTHING global-pip on one machine | venv-per-project (war-avoid!) ✌️ |
| 2 | randomly mixing conda+pip in the same-env | first conda-big, then pip-rest — COLD-ORDER! |
| 3 | tearing up requirements.txt without a freeze | on every change `pip freeze > ...` sync! 📋 |
| 4 | expecting uv everywhere (WSL/termux issues sometimes!) | pip+venv as the reliable-fallback 🌱 |
| 5 | tool-hoarding (setting everything up at once!) | need-driven pick (matrix-rule!) 🗺️ |

> 🎯 **Default-Wisdom:** learn pip+venv first (the foundation!), then a uv-style graduation —
> then move into the professional world of tools! 🚀


## Summary

This notebook mapped the three package power tools. **pip + venv** is the universal default — built into Python, reliable, light; its workflow is create → activate → `pip install` → `pip freeze > requirements.txt` → `deactivate`. **conda** manages environments AND non-Python packages (CUDA, ffmpeg, R libraries), uses `conda create -n ds python=3.11`, and exports to `environment.yml` — the data-science SUV. **uv** is the Rust-based bullet train: 10–100× faster installs, instant `uv venv`, and a modern `pyproject.toml` + lock-file workflow. The tool-pick matrix turns decisions into scenarios: learning/legacy → pip+venv; GPU/CUDA → conda; CI speed and modern monorepos → uv. Finally, the quick-answer drills rehearsed the mapping. Your default wisdom: master pip+venv first, then graduate to the right tool for the job. Next station — file handling and serialization with `json`, `csv`, and `pickle`.